# Задание 2 — «Что произошло с Клиентами»

Показатель **«Доля Клиентов без запчастей»** = доля визитов с выручкой з/ч = 0. Норматив — не более 25%.

В I–II кварталах показатель был выше нормы, в III квартале «кардинально» улучшился. Задача — выяснить причину и показать её графиком.

In [9]:
df['ID Клиента'].value_counts()

ID Клиента
17129    18
16513    13
14408    12
19334    11
18508    11
         ..
13417     1
15447     1
12310     1
17679     1
19504     1
Name: count, Length: 2201, dtype: int64

In [5]:
import pandas as pd
import plotly.express as px

df = pd.read_excel(r'..\task_2.xlsx')
qorder = ['I кв.', 'II кв.', 'III кв.', 'IV кв.']

# Официальный KPI: доля визитов с выручкой з/ч = 0
kpi = df.assign(zero=df['выручка запчастей'] == 0).groupby('квартал')['zero'].mean().reindex(qorder) * 100
print('Доля Клиентов без запчастей (выручка = 0), %:')
print(kpi.round(1))

Доля Клиентов без запчастей (выручка = 0), %:
квартал
I кв.      41.5
II кв.     30.6
III кв.    10.5
IV кв.      4.7
Name: zero, dtype: float64


## Проверка: за счёт чего упал показатель?

Разложим каждый визит на 3 категории по выручке за запчасти:
- **0 руб** — формально «без запчастей»;
- **0–5 руб** — символические (копеечные) начисления, реальной покупкой не являются;
- **≥5 руб** — реальные запчасти.

In [7]:
def cat(x):
    if x == 0:
        return 'Ноль (0 руб)'
    if x < 5:
        return 'Символические (0–5 руб)'
    return 'Реальные з/ч (≥5 руб)'

df['категория'] = df['выручка запчастей'].apply(cat)
comp = df.groupby(['квартал', 'категория']).size().unstack(fill_value=0).reindex(qorder)
comp_pct = comp.div(comp.sum(axis=1), axis=0) * 100
print(comp_pct.round(1).to_string())

catorder = ['Ноль (0 руб)', 'Символические (0–5 руб)', 'Реальные з/ч (≥5 руб)']
plot_df = comp_pct.reset_index().melt(id_vars='квартал', var_name='категория', value_name='доля')
fig = px.bar(
    plot_df, x='квартал', y='доля', color='категория', text_auto='.1f',
    category_orders={'квартал': qorder, 'категория': catorder},
    color_discrete_map={'Ноль (0 руб)': '#d62728',
                        'Символические (0–5 руб)': '#ff7f0e',
                        'Реальные з/ч (≥5 руб)': '#2ca02c'},
    labels={'доля': 'Доля визитов, %'},
    title='Состав чеков по запчастям: нули «перетекают» в копеечные начисления')
fig.show()

категория  Ноль (0 руб)  Реальные з/ч (≥5 руб)  Символические (0–5 руб)
квартал                                                                
I кв.              41.5                   53.5                      5.0
II кв.             30.6                   59.9                      9.5
III кв.            10.5                   71.8                     17.7
IV кв.              4.7                   64.6                     30.6


In [10]:
df

,ID Клиента,Unnamed: 1,дата,выручка запчастей,квартал,категория
0,16301,1,2017-01-03,0.00,I кв.,Ноль (0 руб)
1,11742,1,2017-01-03,113.19,I кв.,Реальные з/ч (≥5 руб)
2,15197,1,2017-01-03,51.76,I кв.,Реальные з/ч (≥5 руб)
3,10767,1,2017-01-03,31.80,I кв.,Реальные з/ч (≥5 руб)
4,14783,1,2017-01-03,42.14,I кв.,Реальные з/ч (≥5 руб)
...,...,...,...,...,...,...
4253,16972,1,2017-12-31,38.78,IV кв.,Реальные з/ч (≥5 руб)
4254,15005,1,2017-12-31,8.20,IV кв.,Реальные з/ч (≥5 руб)
4255,15267,1,2017-12-31,0.00,IV кв.,Ноль (0 руб)
4256,19011,1,2017-12-31,877.29,IV кв.,Реальные з/ч (≥5 руб)


## Доказательство: если считать копейки как «без реальных з/ч»

In [3]:
evid = pd.DataFrame({
    'Доля = 0 (офиц. KPI), %': df.assign(f=df['выручка запчастей'] <= 0).groupby('квартал')['f'].mean().reindex(qorder) * 100,
    'Доля <= 1 руб, %':        df.assign(f=df['выручка запчастей'] <= 1).groupby('квартал')['f'].mean().reindex(qorder) * 100,
    'Доля <= 5 руб, %':        df.assign(f=df['выручка запчастей'] <= 5).groupby('квартал')['f'].mean().reindex(qorder) * 100,
    'Реальные >= 20 руб, %':   df.assign(f=df['выручка запчастей'] >= 20).groupby('квартал')['f'].mean().reindex(qorder) * 100,
}).round(1)
evid

,"Доля = 0 (офиц. KPI), %","Доля <= 1 руб, %","Доля <= 5 руб, %","Реальные >= 20 руб, %"
квартал,,,,
I кв.,41.5,42.4,46.5,48.6
II кв.,30.6,33.4,40.1,54.1
III кв.,10.5,21.2,28.2,64.0
IV кв.,4.7,25.4,35.4,59.3


## Причина

**Показатель «доля без запчастей» упал не потому, что клиенты стали покупать запчасти, а потому, что изменился способ учёта: нулевые чеки заменили символическими копеечными начислениями.**

Что видно из данных:
- Доля нулевых чеков падает 41.5% → 30.6% → 10.5% → 4.7%, но **ровно на столько же растёт доля копеечных начислений 0–5 руб**: 5% → 9.5% → 17.7% → 30.6%. Нули просто перетекли в соседнюю корзину (см. stacked bar).
- В III–IV кв. массово появляются суммы вроде 0.07, 0.13, 0.66 руб — это не покупка запчастей, а формальное начисление, чтобы выручка стала > 0.
- Если «без реальных запчастей» считать как выручку ≤ 1 руб, улучшения почти нет: 42% → 33% → 21% → **25%** (в IV кв. даже отскок вверх).
- Доля с реальной выручкой (≥ 20 руб) выросла лишь умеренно (49% → 54% → 64% → 59%) — далеко не так, как официальные 41.5% → 4.7%.

**Вывод:** это «подкрутка» метрики (gaming KPI), а не реальное изменение поведения клиентов. Клиенты как приезжали со своими запчастями, так и приезжают — просто теперь им пробивают символическую сумму вместо нуля. Чтобы показатель отражал реальность, порог нужно считать не «> 0», а «выше какого-то разумного минимума» (например, ≥ 5–20 руб).